# Teste de Generalização Externa no Corpus FakeRecogna

**Pergunta central:** O modelo treinado no corpus Fake.br generaliza satisfatoriamente para notícias provenientes de outra base de dados independente (FakeRecogna)?

### Contextualização e Cuidados Metodológicos

Ao realizar uma avaliação cruzada de domínios (*cross-corpus*), é indispensável alinhar o formato e a natureza dos textos:
1. **Formato do texto na FakeRecogna:** A coluna `Noticia` da FakeRecogna vem pré-processada de fábrica (lematizada e com remoção de stopwords), enquanto o Fake.br e o pipeline treinado utilizam o texto em linguagem natural. Avaliar notícias lematizadas com um vocabulário aprendido sobre texto bruto geraria incompatibilidade artificial.
2. **Natureza da classe fake:** Os textos marcados como fake na FakeRecogna são, em grande parte, matérias completas produzidas por agências de checagem desmentindo boatos (com expressões como *"É falso que..."*). O boato autêntico está preservado primariamente nos títulos de notícias originárias do veículo `boatos.org` (onde o padrão é `<boato> #boato`).
3. **Estratégia de teste adotada:**
   - Selecionam-se os títulos da classe fake provenientes do domínio `boatos.org`, removendo a tag indicativa `#boato`.
   - Amostra-se uma quantidade idêntica de títulos de notícias verdadeiras (balanceamento 50/50).
   - Aplica-se a **mesma normalização de caracteres e dígitos** definida no `01_Data_Prep.ipynb` (operação pura por documento, sem vazamento).
   - O classificador é treinado no conjunto `X_tr` completo do Fake.br e avaliado diretamente nos títulos da FakeRecogna. O conjunto de teste original (`X_te`) segue intocado.

In [ ]:
# Resolve input and output paths from the supervised research root.
from pathlib import Path
import sys

_cwd = Path.cwd().resolve()
_candidates = [candidate for parent in (_cwd, *_cwd.parents)
               for candidate in (parent, parent / "machine-learning/supervised-learning")]
ROOT = next((candidate for candidate in _candidates
             if (candidate / "modelo_olimpo.py").is_file()
             and (candidate / "history/baselines/01_Data_Prep.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the Olimpo repository.")
DATA = ROOT / "data"
HISTORY_RESULTS = ROOT / "history/resultados"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Original experiment starts here.
import os
import re
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, roc_auc_score

# Carregamento dos dados Fake.br
X_tr, X_te, y_tr, y_te = joblib.load((DATA / 'dados_preparados.pkl'))
configs = joblib.load((DATA / 'transformers.pkl'))

# Carregamento / Download do dataset FakeRecogna
CSV = Path((DATA / 'FakeRecogna.csv'))
if not CSV.exists():
    print("Baixando FakeRecogna.csv...")
    pd.read_csv("https://huggingface.co/datasets/recogna-nlp/FakeRecogna/resolve/main/FakeRecogna.csv").to_csv(CSV, index=False)
else:
    print(f"Dataset FakeRecogna encontrado ({CSV.stat().st_size / (1024*1024):.1f} MB).")

## Pré-Processamento e Montagem da Base de Teste Externo

Implementamos a função `normalizar` idêntica à do `01_Data_Prep.ipynb` e estruturamos a base balanceada de títulos.

In [ ]:
def normalizar(texto):
    texto = str(texto)
    texto = texto.translate(str.maketrans({"\x96": "-", "\x97": "-", "\x93": '"', "\x94": '"', "\x91": "'", "\x92": "'"}))
    texto = re.sub(r"[\x80-\x9f]", " ", texto)
    texto = re.sub(r"[–—]", "-", texto)
    texto = re.sub(r"[“”«»]", '"', texto)
    texto = re.sub(r"[‘’]", "'", texto)
    texto = re.sub(r"\s*(?:\[\.*\]|\.{2,}|…)", ".", texto)
    texto = re.sub(r"\d", "0", texto)
    return " ".join(texto.split())

d = pd.read_csv(CSV)
d["dominio"] = d.URL.fillna("").str.extract(r"https?://(?:www\.)?([^/]+)")[0]

# Extração dos boatos autênticos do boatos.org (Classe 0 no FakeRecogna = Fake)
fake = d[(d.Classe == 0) & (d.dominio == "boatos.org")].copy()
fake["texto_trunc"] = fake.Titulo.str.replace(r"#boato", "", case=False, regex=True)

# Amostragem pareada de notícias reais (Classe 1 no FakeRecogna = Verdadeira)
real = d[d.Classe == 1].sample(len(fake), random_state=42).copy()
real["texto_trunc"] = real.Titulo

# Concatenação e normalização
ext = pd.concat([fake, real])
ext["texto_trunc"] = ext.texto_trunc.fillna("").map(normalizar)
ext = ext[ext.texto_trunc.str.len() > 0]
ext["label"] = (ext.Classe == 0).astype(int)  # Alinhamento de rótulos: 1 = fake, 0 = real (padrão Fake.br)

print(f"Conjunto de Teste Externo: {len(ext):,} títulos ({ext.label.sum():,} fake / {(1 - ext.label).sum():,} real)")
print(f"Extensão mediana: {ext.texto_trunc.str.split().str.len().median():.0f} palavras por título")

## Treinamento no Fake.br e Avaliação no FakeRecogna

Avaliamos o impacto de diferentes configurações de regularização ($C$) e diferentes níveis de poda de vocabulário (`min_df`) para verificar se modelos mais conservadores generalizam melhor fora do domínio.

Os resultados são salvos em `../resultados/resultados_teste_externo.csv`.

In [ ]:
modelos = {
    "SVC": LinearSVC(class_weight="balanced", random_state=42, max_iter=5000),
    "LR":  LogisticRegression(max_iter=2000, class_weight="balanced"),
}

cenarios = [(C, 2, 3) for C in [0.01, 0.03, 0.1, 0.3, 1, 3, 10]] + [(1, mw, mc) for mw, mc in [(1, 1), (5, 5), (10, 10)]]
csv_externo = (HISTORY_RESULTS / 'resultados_teste_externo.csv')

if os.path.exists(csv_externo):
    print(f"Carregando resultados pré-calculados de {csv_externo}...")
    res_ext = pd.read_csv(csv_externo)
else:
    print("Executando treinamento no Fake.br e avaliação no FakeRecogna...")
    linhas = []
    for nome, clf in modelos.items():
        for C, mw, mc in cenarios:
            pipe = Pipeline([
                ("prep", ColumnTransformer(configs["word+char"])),
                ("clf", clone(clf).set_params(C=C))
            ])
            pipe.set_params(prep__txt_word__min_df=mw, prep__txt_char__min_df=mc)
            pipe.fit(X_tr, y_tr)
            
            score = pipe.decision_function(ext[["texto_trunc"]])
            pred = pipe.predict(ext[["texto_trunc"]])
            
            linhas.append({
                "modelo": nome,
                "C": C,
                "min_df_word": mw,
                "min_df_char": mc,
                "f1_externo": f1_score(ext.label, pred, average="macro"),
                "auc_externo": roc_auc_score(ext.label, score),
                "pct_prev_fake": float(pred.mean())
            })
            print(f"{nome:<4} | C={C:<5} | min_df=({mw},{mc}) | F1={linhas[-1]['f1_externo']:.4f} | AUC={linhas[-1]['auc_externo']:.4f} | Prev. Fake={linhas[-1]['pct_prev_fake']:.2%}", flush=True)
    res_ext = pd.DataFrame(linhas)
    res_ext.to_csv(csv_externo, index=False)

## Tabelas Comparativas de Desempenho Externo

In [ ]:
print("=== Resultados no Teste Externo (Ordenados por Macro F1) ===")
display(res_ext.sort_values("f1_externo", ascending=False).round(4).reset_index(drop=True))

## Visualização Gráfica do Desempenho Externo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subconjunto com min_df padrão (2, 3) variando C
sub_c = res_ext[(res_ext.min_df_word == 2) & (res_ext.min_df_char == 3)].sort_values("C")

for modelo in ["SVC", "LR"]:
    s = sub_c[sub_c.modelo == modelo]
    axes[0].plot(s["C"], s["f1_externo"], marker="o", lw=2, label=f"{modelo} - F1 Macro")
    axes[0].plot(s["C"], s["auc_externo"], marker="s", lw=2, linestyle="--", label=f"{modelo} - ROC AUC")

axes[0].set_xscale("log")
axes[0].set_title("Desempenho Externo vs Parâmetro C", fontsize=12)
axes[0].set_xlabel("C (escala log)", fontsize=10)
axes[0].set_ylabel("Métrica", fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=9)

# Percentual previsto como fake
for modelo in ["SVC", "LR"]:
    s = sub_c[sub_c.modelo == modelo]
    axes[1].plot(s["C"], s["pct_prev_fake"] * 100, marker="^", lw=2, label=f"{modelo}")

axes[1].axhline(50, color="gray", linestyle=":", label="Proporção Real (50%)")
axes[1].set_xscale("log")
axes[1].set_title("Viés de Decisão: % Previsto como Fake", fontsize=12)
axes[1].set_xlabel("C (escala log)", fontsize=10)
axes[1].set_ylabel("% Previsto como Fake", fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Teste de Redução de Dimensionalidade (PCA / TruncatedSVD e $\chi^2$) no FakeRecogna

**Hipótese formulada:** Um F1 alto na validação cruzada interna (Fake.br) não significa necessariamente que o modelo seja melhor para o mundo real. Modelos com 114 mil features podem memorizar artefatos e peculiaridades de estilo exclusivas do Fake.br (*dataset artifacts*), sofrendo *overfitting* de domínio e apresentando viés severo quando avaliados em outro corpus.

Testamos como a redução de dimensionalidade densa (**PCA / TruncatedSVD** com $d \in [2, 5, 10, 20, 50, 100, 200, 300, 500]$) e esparsa (**$\chi^2$** com $k \in [5000, 10000, 20000, 50000, 80000]$) se comporta no conjunto externo da FakeRecogna.

In [ ]:
res_dim = pd.read_csv((HISTORY_RESULTS / 'resultados_externo_pca.csv'))
print("=== Comparativo de Dimensionalidade no Teste Externo (Ordenado por F1 Externo) ===")
display(res_dim.sort_values("f1_externo", ascending=False).round(4).reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. F1 Externo vs Dimensionalidade
svd_ext = res_dim[res_dim.modelo.str.contains("PCA/SVD")].sort_values("n_dim")
chi2_ext = res_dim[res_dim.modelo.str.contains("Chi2")].sort_values("n_dim")
base_ext_f1 = res_dim[res_dim.modelo.str.contains("Base")]["f1_externo"].iloc[0]

axes[0].plot(svd_ext["n_dim"], svd_ext["f1_externo"], marker="s", lw=2, color="#eb6834", label="PCA / TruncatedSVD")
axes[0].plot(chi2_ext["n_dim"], chi2_ext["f1_externo"], marker="o", lw=2, color="#2a78d6", label=r"$\chi^2$ (SelectKBest)")
axes[0].axhline(base_ext_f1, color="#222222", linestyle="--", lw=1.5, label="Base (~114k features)")

axes[0].set_xscale("log")
axes[0].set_title("F1 Macro no Teste Externo vs Dimensões (Escala Log)", fontsize=12)
axes[0].set_xlabel("Número de Dimensões / Features (log)", fontsize=10)
axes[0].set_ylabel("F1 Macro Externo", fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=9)

# 2. Viés de Predição (% Previsto como Fake)
axes[1].plot(svd_ext["n_dim"], svd_ext["pct_prev_fake"] * 100, marker="s", lw=2, color="#eb6834", label="PCA / TruncatedSVD")
axes[1].plot(chi2_ext["n_dim"], chi2_ext["pct_prev_fake"] * 100, marker="o", lw=2, color="#2a78d6", label=r"$\chi^2$ (SelectKBest)")
axes[1].axhline(50, color="green", linestyle=":", lw=2, label="Equilíbrio Real (50%)")
axes[1].axhline(res_dim[res_dim.modelo.str.contains("Base")]["pct_prev_fake"].iloc[0] * 100, color="#222222", linestyle="--", label="Base (~71% fake)")

axes[1].set_xscale("log")
axes[1].set_title("Viés de Decisão Externa: % Previsto como Fake", fontsize=12)
axes[1].set_xlabel("Número de Dimensões (log)", fontsize=10)
axes[1].set_ylabel("% Previsto como Fake", fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Conclusões sobre a Generalização Externa

1. **A 'Ilusão' do F1 Interno:**
   - Na validação cruzada interna (Fake.br), a representação completa de 114k features vence com folga (**F1 = 0.9295**).
   - **Porém, no dataset externo (FakeRecogna), o modelo de 114k features sofre severamente de viés de domínio**, prevendo fake em **70.7%** das vezes e obtendo F1 de apenas **0.6122**.
2. **A Redução de Dimensionalidade Generaliza Muito Melhor Fora do Domínio:**
   - Ao comprimir o espaço via **PCA / TruncatedSVD ($d=300$ ou $500$)**, o F1 externo salta para **0.6439 - 0.6518** (+0.040 sobre a base!), e o viés de predição cai de 70.7% para **~55%**!
   - Mesmo com **$d=50$**, o modelo já atinge **F1 = 0.6278** com apenas 54.6% de predições fake, superando a base de 114k features.
   - Com **$\chi^2$ ($k=20.000$)**, o F1 externo também sobe para **0.6451**.
3. **Mecanismo Subjacente:**
   - Ao limitar as dimensões ($d < N$), o modelo é **impedido de decorar n-gramas e vícios de estilo específicos do corpus Fake.br**.
   - A compressão semântica atua como uma forte regularização estrutural, forçando o classificador a tomar decisões baseadas em padrões semânticos latentes mais universais.